### 1. Chargement des données brutes  

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

DATA_PATH = Path('../data/raw/heart_disease_uci.csv')
PROC_PATH = Path('../data/processed')
FIGS_PATH = Path('../reports/figures')

# Colonnes par type
NUM_COLS = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca', 'chol_per_age', 'thalch_ratio']
BIN_COLS = ['fbs', 'exang', 'ca_missing', 'thal_missing']
OHE_COLS = ['cp', 'thal', 'dataset']
LBL_COLS = ['sex', 'restecg', 'slope']
TARGET   = 'num'

df = pd.read_csv(DATA_PATH)
df = df.drop(columns=['id'])
print(f'Données chargées : {df.shape[0]} lignes × {df.shape[1]} colonnes')

Données chargées : 920 lignes × 15 colonnes


### 2 — Nettoyage des valeurs aberrantes

In [ ]:
df_before = df.copy()

# chol = 0 → NaN
n_chol = (df['chol'] == 0).sum()
df.loc[df['chol'] == 0, 'chol'] = np.nan

# trestbps = 0 → NaN
n_bps = (df['trestbps'] == 0).sum()
df.loc[df['trestbps'] == 0, 'trestbps'] = np.nan

# oldpeak < 0 → 0
n_old = (df['oldpeak'] < 0).sum()
df.loc[df['oldpeak'] < 0, 'oldpeak'] = 0.0

print(f'Corrections appliquées :')
print(f'  chol = 0     → NaN : {n_chol} valeurs')
print(f'  trestbps = 0 → NaN : {n_bps} valeur')
print(f'  oldpeak < 0  → 0   : {n_old} valeurs')

# Visualisation avant/après pour chol
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df_before['chol'].dropna(), bins=30, color='#e74c3c', edgecolor='white', alpha=0.8)
axes[0].set_title('chol — AVANT nettoyage  (0 inclus)')
axes[0].set_xlabel('Cholestérol (mg/dl)')
axes[1].hist(df['chol'].dropna(), bins=30, color='#2ecc71', edgecolor='white', alpha=0.8)
axes[1].set_title('chol — APRÈS nettoyage  (0 → NaN supprimés)')
axes[1].set_xlabel('Cholestérol (mg/dl)')
plt.tight_layout()
plt.savefig(FIGS_PATH / '08_chol_before_after.png', dpi=150, bbox_inches='tight')
plt.show()